In [1]:
import numpy as np
from scipy import stats

# Практика 14. Параметричні критерії
# Варіант 1 — Київ

city = "Київ"
variant = 1
norm = 74
alpha = 0.05

morning = np.array([
    84, 82, 78, 70, 65, 63,
    61, 63, 68, 75, 81, 85
])

months = [
    "Січень", "Лютий", "Березень", "Квітень",
    "Травень", "Червень", "Липень", "Серпень",
    "Вересень", "Жовтень", "Листопад", "Грудень"
]

n = len(morning)
x_bar = morning.mean()
s = morning.std(ddof=1)

print("Варіант:", variant)
print("Місто:", city)
print("Норма вологості:", norm, "%")
print("Місячні значення:", morning)
print("Кількість спостережень:", n)
print("Середнє:", round(x_bar, 4), "%")
print("Стандартне відхилення:", round(s, 4))

# Завдання 1. Одновибірковий t-критерій
t_stat, p_value = stats.ttest_1samp(morning, popmean=norm)

print("\nt-статистика:", round(t_stat, 4))
print("p-value:", round(p_value, 4))

if p_value < alpha:
    print("Відхиляємо H0: середня вологість відрізняється від норми.")
else:
    print("Не відхиляємо H0: недостатньо доказів відмінності від норми.")

Варіант: 1
Місто: Київ
Норма вологості: 74 %
Місячні значення: [84 82 78 70 65 63 61 63 68 75 81 85]
Кількість спостережень: 12
Середнє: 72.9167 %
Стандартне відхилення: 8.9489

t-статистика: -0.4194
p-value: 0.683
Не відхиляємо H0: недостатньо доказів відмінності від норми.


In [2]:
# Завдання 2. Ручний розрахунок t-статистики

t_manual = (x_bar - norm) / (s / np.sqrt(n))

print("t-статистика вручну:", round(t_manual, 6))
print("t-статистика scipy:", round(t_stat, 6))
print("Результати збігаються:", np.isclose(t_manual, t_stat))

t-статистика вручну: -0.419355
t-статистика scipy: -0.419355
Результати збігаються: True


In [3]:
# Завдання 3. Незалежний двовибірковий t-критерій
# Київ — Центр, Львів — Захід

lviv = np.array([
    85, 83, 80, 73, 68, 65,
    63, 65, 70, 77, 82, 86
])

# Спочатку перевіряємо рівність дисперсій
stat_lev_1, p_lev_1 = stats.levene(morning, lviv)

print("Критерій Лівена:")
print("Статистика:", round(stat_lev_1, 4))
print("p-value:", round(p_lev_1, 4))

equal_var_1 = p_lev_1 >= alpha

if equal_var_1:
    print("Немає достатніх доказів нерівності дисперсій.")
    print("Використовуємо стандартний t-критерій.")
else:
    print("Дисперсії статистично відрізняються.")
    print("Використовуємо критерій Welch.")

t_ind_1, p_ind_1 = stats.ttest_ind(
    morning, lviv, equal_var=equal_var_1
)

print("\nПорівняння середньої вологості Києва та Львова:")
print("Середнє Києва:", round(morning.mean(), 2), "%")
print("Середнє Львова:", round(lviv.mean(), 2), "%")
print("t-статистика:", round(t_ind_1, 4))
print("p-value:", round(p_ind_1, 4))
print("equal_var:", equal_var_1)

if p_ind_1 < alpha:
    print("Середня вологість міст статистично значуще відрізняється.")
else:
    print("Немає достатніх доказів відмінності середньої вологості міст.")

Критерій Лівена:
Статистика: 0.129
p-value: 0.7229
Немає достатніх доказів нерівності дисперсій.
Використовуємо стандартний t-критерій.

Порівняння середньої вологості Києва та Львова:
Середнє Києва: 72.92 %
Середнє Львова: 74.75 %
t-статистика: -0.5162
p-value: 0.6109
equal_var: True
Немає достатніх доказів відмінності середньої вологості міст.


In [4]:
# Завдання 4. Пошук міста з протилежним результатом Лівена

cities = {
    "Одеса": (np.array([82, 80, 78, 74, 71, 69, 67, 68, 72, 76, 80, 83]), "Південь"),
    "Харків": (np.array([83, 80, 75, 68, 62, 60, 58, 60, 66, 73, 79, 84]), "Схід"),
    "Дніпро": (np.array([82, 79, 74, 67, 61, 59, 57, 59, 65, 72, 78, 83]), "Схід"),
    "Чернігів": (np.array([85, 83, 79, 71, 66, 63, 61, 63, 69, 76, 82, 86]), "Центр"),
    "Івано-Франківськ": (np.array([86, 84, 81, 75, 70, 67, 65, 67, 72, 78, 83, 87]), "Захід"),
    "Полтава": (np.array([84, 81, 76, 69, 63, 61, 59, 61, 67, 74, 80, 85]), "Центр"),
    "Суми": (np.array([85, 82, 77, 70, 64, 62, 60, 62, 68, 75, 81, 86]), "Схід"),
    "Ужгород": (np.array([84, 82, 79, 73, 68, 66, 64, 66, 71, 77, 82, 85]), "Захід")
}

target_result = p_lev_1 < alpha
selected_city = None

for name, (values, region) in cities.items():
    if region == "Центр" or region == "Захід":
        continue

    stat_lev, p_lev = stats.levene(morning, values)
    print(name, "| p-value Лівена:", round(p_lev, 4))

    if (p_lev < alpha) != target_result:
        selected_city = name
        selected_values = values
        selected_p_lev = p_lev
        break

if selected_city is not None:
    equal_var_2 = selected_p_lev >= alpha
    t_ind_2, p_ind_2 = stats.ttest_ind(
        morning, selected_values, equal_var=equal_var_2
    )

    print("\nОбране місто:", selected_city)
    print("p-value Лівена:", round(selected_p_lev, 4))
    print("equal_var:", equal_var_2)
    print("t-статистика:", round(t_ind_2, 4))
    print("p-value t-критерію:", round(p_ind_2, 4))

    if p_ind_2 < alpha:
        print("Середня вологість статистично значуще відрізняється.")
    else:
        print("Недостатньо доказів відмінності середньої вологості.")
else:
    print("Місто з протилежним результатом не знайдено.")

Одеса | p-value Лівена: 0.0198

Обране місто: Одеса
p-value Лівена: 0.0198
equal_var: False
t-статистика: -0.6828
p-value t-критерію: 0.5032
Недостатньо доказів відмінності середньої вологості.


In [5]:
# Завдання 5. Парний t-критерій: ранок проти вечора

amplitude = np.array([
    3, 3, 4, 5, 6, 7,
    7, 7, 6, 5, 4, 3
])

evening = morning - amplitude

t_rel, p_rel = stats.ttest_rel(morning, evening)

print("Ранкова вологість:", morning)
print("Вечірня вологість:", evening)
print("Середня вологість вранці:", round(morning.mean(), 2), "%")
print("Середня вологість увечері:", round(evening.mean(), 2), "%")
print("Середня різниця:", round((morning - evening).mean(), 2), "п.п.")
print("t-статистика:", round(t_rel, 4))
print("p-value:", round(p_rel, 6))

if p_rel < alpha:
    print("Різниця між ранковою та вечірньою вологістю статистично значуща.")
else:
    print("Недостатньо доказів статистично значущої різниці.")

Ранкова вологість: [84 82 78 70 65 63 61 63 68 75 81 85]
Вечірня вологість: [81 79 74 65 59 56 54 56 62 70 77 82]
Середня вологість вранці: 72.92 %
Середня вологість увечері: 67.92 %
Середня різниця: 5.0 п.п.
t-статистика: 10.8562
p-value: 0.0
Різниця між ранковою та вечірньою вологістю статистично значуща.


## Відповіді на контрольні запитання

**1. Чому використовується t-розподіл, а не z-розподіл?**

Тому що стандартне відхилення генеральної сукупності невідоме, і ми оцінюємо його за вибіркою.

**2. Чому важливо правильно обирати `equal_var`?**

Якщо дисперсії різні, стандартний критерій може давати ненадійне значення p-value. Тому спочатку перевіряємо дисперсії критерієм Лівена.

**3. Чому парний критерій може бути потужнішим?**

Він враховує зв’язок між вимірюваннями одного місяця. Різниці для кожної пари усувають вплив спільних місячних особливостей.

**4. Чи збігається результат Лівена з порівнянням стандартних відхилень?**

Стандартні відхилення допомагають оцінити розкид на око, але критерій Лівена формально перевіряє, чи є статистичні підстави вважати дисперсії різними.